# 👁️ Day 06: Tree Views — Seeing Trees from Different Angles

---

## The "Views" Family

These problems ask you to see the tree from a specific perspective: right side, left side, top, bottom. They all use BFS (level order) or DFS with clever tracking.

```
         1
        / \
       2   3        Right view:  [1, 3, 4]  (last node per level)
        \   \       Left view:   [1, 2, 4]  (first node per level)
         5   4
```

---

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

from collections import deque, defaultdict

def build_tree(values):
    if not values:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

## 🎯 Problem 1: Binary Tree Right Side View (LC #199)

Imagine standing on the RIGHT side of the tree. You see the last node of each level.

**BFS approach:** Level order traversal, take the last node of each level.

In [ ]:
def rightSideView(root):
    """
    BFS: last element of each level.
    Time: O(n), Space: O(n)
    """
    if not root:
        return []
    
    result = []
    queue = deque([root])
    
    while queue:
        level_size = len(queue)
        for i in range(level_size):
            node = queue.popleft()
            if i == level_size - 1:  # Last node in level
                result.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
    
    return result

tree = build_tree([1, 2, 3, None, 5, None, 4])
print(f"Right side view: {rightSideView(tree)}")  # [1, 3, 4]

**DFS alternative:** Go right first, track depth. First node at each new depth is the right side view.

In [ ]:
def rightSideView_DFS(root):
    """DFS: right first, first node at each depth."""
    result = []
    
    def dfs(node, depth):
        if not node:
            return
        if depth == len(result):  # First time at this depth
            result.append(node.val)
        dfs(node.right, depth + 1)  # Right first!
        dfs(node.left, depth + 1)
    
    dfs(root, 0)
    return result

print(f"Right side (DFS): {rightSideView_DFS(tree)}")  # [1, 3, 4]

---

## 🎯 Problem 2: Vertical Order Traversal (LC #987)

Assign each node a column: root = 0, left child = col-1, right child = col+1. Group by column, sort within same position.

```
         3 (col 0)
        / \
   (col-1)  (col+1)
      9       20
             / \
        (col 0) (col+2)
           15     7

Output: [[9], [3,15], [20], [7]]
```

In [ ]:
def verticalTraversal(root):
    """
    BFS with (col, row) tracking. Sort by col → row → val.
    Time: O(n log n), Space: O(n)
    """
    if not root:
        return []
    
    # Collect (col, row, val) for all nodes
    nodes = []
    queue = deque([(root, 0, 0)])  # (node, col, row)
    
    while queue:
        node, col, row = queue.popleft()
        nodes.append((col, row, node.val))
        if node.left:
            queue.append((node.left, col - 1, row + 1))
        if node.right:
            queue.append((node.right, col + 1, row + 1))
    
    # Sort by column, then row, then value
    nodes.sort()
    
    # Group by column
    result = []
    prev_col = float('-inf')
    for col, row, val in nodes:
        if col != prev_col:
            result.append([])
            prev_col = col
        result[-1].append(val)
    
    return result

tree = build_tree([3, 9, 20, None, None, 15, 7])
print(f"Vertical order: {verticalTraversal(tree)}")  # [[9], [3, 15], [20], [7]]

---

## 🎯 Problem 3: Binary Tree Level Order as Flat + Boundary Traversal

Boundary traversal = left boundary + leaves + right boundary (in reverse). Popular in interviews at Amazon/Microsoft.

In [ ]:
def boundaryOfBinaryTree(root):
    """
    Left boundary (top-down) + Leaves (left-to-right) + Right boundary (bottom-up).
    """
    if not root:
        return []
    
    boundary = [root.val]
    
    def is_leaf(node):
        return node and not node.left and not node.right
    
    # Left boundary (exclude leaf)
    def left_boundary(node):
        if not node or is_leaf(node):
            return
        boundary.append(node.val)
        if node.left:
            left_boundary(node.left)
        else:
            left_boundary(node.right)
    
    # All leaves left to right
    def leaves(node):
        if not node:
            return
        if is_leaf(node):
            boundary.append(node.val)
            return
        leaves(node.left)
        leaves(node.right)
    
    # Right boundary (exclude leaf, added in reverse)
    def right_boundary(node):
        if not node or is_leaf(node):
            return
        if node.right:
            right_boundary(node.right)
        else:
            right_boundary(node.left)
        boundary.append(node.val)  # Add AFTER recursion = reverse order
    
    left_boundary(root.left)
    leaves(root.left)
    leaves(root.right)
    right_boundary(root.right)
    
    return boundary

tree = build_tree([1, 2, 3, 4, 5, 6, 7, None, None, 8, 9])
print(f"Boundary: {boundaryOfBinaryTree(tree)}")
# [1, 2, 4, 8, 9, 6, 7, 3]

---

## 🗺️ Tree Views Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TREE VIEW PATTERNS                                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. RIGHT/LEFT VIEW: BFS, last/first of each level                  ║
║     DFS alt: go right/left first, depth == len(result) = new level  ║
║                                                                      ║
║  2. VERTICAL ORDER: Track (col, row), sort, group by column         ║
║                                                                      ║
║  3. BOUNDARY: 3 parts — left edge + leaves + right edge reversed   ║
║                                                                      ║
║  COMMON TECHNIQUE: BFS with extra metadata (col, row, depth)        ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 199 | Right Side View | Medium | BFS: last per level |
| 987 | Vertical Order Traversal | Hard | col/row tracking + sort |
| 545 | Boundary of Binary Tree | Medium | Left + leaves + right(rev) |
| 314 | Binary Tree Vertical Order | Medium | Simpler vertical (no sort) |
| 116 | Populate Next Right Pointers | Medium | Level-order linking |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Right/Left view = BFS + pick first/last of each level         │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Vertical order: assign columns (left = col-1, right = col+1)  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Boundary: 3 separate traversals combined                      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 DFS view trick: depth == len(result) means new level found    │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Trie (Prefix Tree)

A completely different kind of tree — for string/prefix problems.